# Bekami — adaptateur RAG/agent sur Kaggle (piste B)
Même expérience que `Bekami_Colab_RAG_Agent.ipynb`, avec le code cloné depuis GitHub.

**Réglages Kaggle avant de lancer** (panneau de droite, *Session options*) :
- **Accelerator : GPU T4 x2** (évitez P100 : `bitsandbytes` y est moins fiable).
- **Internet : On** (nécessaire pour cloner GitHub et télécharger le modèle ; Kaggle exige un compte vérifié par téléphone).

Un bon score au banc prouve que le modèle a appris le **format et la discipline de citation**, pas qu'il comprend des documents réels. Mesurez toujours le modèle de base d'abord.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Activez un GPU : Session options > Accelerator'
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))

## 1. Projet depuis GitHub
Le dépôt doit contenir les dernières modifications (`git push` depuis votre Mac) : sinon Kaggle récupère une ancienne version.

In [ ]:
import os
if not os.path.isdir('/kaggle/working/afrimind_llm'):
    !git clone --depth 1 https://github.com/rg4amia/afrimind_llm.git /kaggle/working/afrimind_llm
%cd /kaggle/working/afrimind_llm
!git pull -q
!git log --oneline -1
!pip install -q -r requirements-ft.txt

## 2. Données et banc
Régénérés de façon déterministe ; le banc doit afficher 100 % avec les réponses de référence.

In [ ]:
!python build_rag_sft.py --n-train 1500
!python evaluate_rag.py --oracle
!grep base_model adapters/rag-agent/adapter.yaml

## 3. Mesure de référence (modèle de base, avant entraînement)

In [ ]:
!python evaluate_rag.py --hf Qwen/Qwen2.5-1.5B-Instruct --out /kaggle/working/base_rag.json

## 4. Entraînement de l'adaptateur
Le modèle doit être le même que `base_model` dans `adapters/rag-agent/adapter.yaml`.

Les points de reprise vont dans `/kaggle/working/runs`. Kaggle ne conserve `/kaggle/working` qu'après **Save Version** : si la session s'arrête avant, ils sont perdus. L'entraînement (~1 h) tient dans une session de 9 h ; lancez-le en *Save & Run All (Commit)* pour garder les résultats même si vous fermez la page.

In [ ]:
!python finetune.py rag-agent --runs-dir /kaggle/working/runs

## 5. Mesure avec l'adaptateur

In [ ]:
!python evaluate_rag.py --hf Qwen/Qwen2.5-1.5B-Instruct --adapter rag-agent --out /kaggle/working/tuned_rag.json

## 6. Comparaison par comportement

In [ ]:
import json
base, tuned = json.load(open('/kaggle/working/base_rag.json')), json.load(open('/kaggle/working/tuned_rag.json'))
print(f"{'comportement':<14}{'base':>8}{'adaptateur':>12}")
for k in tuned['by_kind']:
    print(f"{k:<14}{base['by_kind'][k]:>8.0%}{tuned['by_kind'][k]:>12.0%}")
print(f"{'GLOBAL':<14}{base['score']:>8.0%}{tuned['score']:>12.0%}   ({tuned['n']} cas)")
print('Chaque type compte 8 à 40 cas : un écart de quelques points est dans le bruit.')

## 7. Adaptateur final
Copié dans `/kaggle/working/bekami_rag-agent_weights`, téléchargeable depuis l'onglet **Output** après *Save Version*.

In [ ]:
!cp -r adapters/rag-agent/weights /kaggle/working/bekami_rag-agent_weights
!ls -lh /kaggle/working/bekami_rag-agent_weights